# Notebook to create NaiNUQ training dataset

## Data download
NANUK simulation can be found on this [storage cloud](https://ige-meom-opendap.univ-grenoble-alpes.fr/thredds/catalog/meomopendap/extract/MEOM/simus_nanuq/NANUK1/old/NANUK1-N1CPL00-S/catalog.html)

Note that in the folder there is several simulations from NANUK available:
Everytime, 

- `NANUK1-N1CPL00-S` : 2010-2020 simulation (2009 is used as one year of spin-up). In this simulation, the velocities of the first layer of the ocean is solved under two name vozocrtx`, `vomecrty`
- `NANUK1-CPL00-S` : 2010-2020 simulation (2009 is used as one year of spin-up)


# Utils

In [1]:
from itertools import product
from scipy.spatial import cKDTree
import xarray as xr
import argparse
import numpy as np
import zarr
from tqdm import trange
from tqdm.auto import tqdm
import os
from glob import glob
import gc
import sys
import matplotlib.pyplot as plt
sys.argv = [""]
import pandas as pd
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior()
import matplotlib.pyplot as plt

2025-11-14 08:43:41.681176: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-14 08:43:44.515556: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-11-14 08:43:44.515587: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
def parse_args():
    parser = argparse.ArgumentParser(description='Create dataset')
    parser.add_argument('--train_year_begin', type=int, default=2010)
    parser.add_argument('--train_year_end', type=int, default=2017)
    parser.add_argument('--val_year_begin', type=int, default=2018)
    parser.add_argument('--val_year_end', type=int, default=2018)
    parser.add_argument('--test_year_begin', type=int, default=2019)
    parser.add_argument('--test_year_end', type=int, default=2020)
    parser.add_argument('--months_begin', type=int, default=1)
    parser.add_argument('--months_end', type=int, default=12)
    parser.add_argument('--save_dir', type=str, default='/summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/')
    parser.add_argument('--path_to_atmo_forcings', type=str, default = '/summer/sasip/model-forcings/atmo_forcing/ERA5_full/')
    parser.add_argument('--SIM_dir', type=str, default='/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/')
    parser.add_argument('--mask_path', type=str, default='mask_SIL_regions.npy')
    parser.add_argument('--y_min', type=int, default=0)
    parser.add_argument('--y_max', type=int, default=0)
    parser.add_argument('--x_min', type=int, default=0)
    parser.add_argument('--x_max', type=int, default=0)
    parser.add_argument('--timestep_difference', type=int, default=6)
    parser.add_argument(
        "--sea_ice_variable",
        nargs="+",
        default=["sivolu", "siconc", "u_ice", "v_ice", "snvolu"],
        help="List of sea ice names"
    )
    parser.add_argument(
        "--atmo_variable",
        nargs="+",
        default=[ "t2m", "mtpr", "d2m", "msdlwrf", "msdswrf"],
        help="List of ERA5 atmospheric variables"
    )
    parser.add_argument(
        "--ocean_variable",
        nargs="+",
        default=["zos", "tos", "sos"],
        help="List of ERA5 atmospheric variables"
    )
    parser.add_argument(
        "--ocean_forcings",
        nargs="+",
        default=["vozocrtx", "vomecrty"],
        help="List of ERA5 atmospheric variables"
    )
    return parser.parse_args()

args = parse_args()


In [8]:
nanuk = xr.open_mfdataset('/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/00*_dev/NANUK1-N1CPL00_1h_*_ovel30.nc4')
nanuk
#print(np.shape(nanuk.time_counter.data))

<xarray.Dataset>
Dimensions:               (y_grid_U: 129, x_grid_U: 118, deptht30: 3,
                           time_counter: 97896, axis_nbounds: 2, y_grid_V: 129,
                           x_grid_V: 118)
Coordinates:
    nav_lat_grid_U        (y_grid_U, x_grid_U) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    nav_lon_grid_U        (y_grid_U, x_grid_U) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
  * deptht30              (deptht30) float32 22.76 26.56 30.87
    nav_lat_grid_V        (y_grid_V, x_grid_V) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    nav_lon_grid_V        (y_grid_V, x_grid_V) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    time_centered         (time_counter) datetime64[ns] dask.array<chunksize=(1464,), meta=np.ndarray>
  * time_counter          (time_counter) datetime64[ns] 2009-11-01T00:30:00 ....
Dimensions without coordinates: y_grid_U, x_grid_U, axis_nbounds, y_grid_V,
                                x_grid_V
Data variables:
    deptht30_bounds       (time_counter, deptht30, axis_nbounds) float32 dask.array<chunksize=(1464, 3, 2), meta=np.ndarray>
    time_centered_bounds  (time_counter, axis_nbounds) datetime64[ns] dask.array<chunksize=(1464, 2), meta=np.ndarray>
    time_counter_bounds   (time_counter, axis_nbounds) datetime64[ns] dask.array<chunksize=(1464, 2), meta=np.ndarray>
    vozocrtx              (time_counter, deptht30, y_grid_U, x_grid_U) float32 dask.array<chunksize=(1464, 3, 129, 118), meta=np.ndarray>
    vomecrty              (time_counter, deptht30, y_grid_V, x_grid_V) float32 dask.array<chunksize=(1464, 3, 129, 118), meta=np.ndarray>
Attributes:
    name:         /workdir/meom/brodeau/NANUK1/NANUK1-N1CPL00-S/00000001-0000...
    description:  ocean velocity
    title:        ocean velocity
    Conventions:  CF-1.6
    timeStamp:    2025-Jun-19 07:38:22 GMT
    uuid:         d5e77d00-fa19-49ce-ae67-7837f028257c

# Functions

In [17]:
def create_folder_structure(args):
    # Check if the main folder exists, if not create it
    if not os.path.exists(args.save_dir):
        os.makedirs(args.save_dir)
        print(f"Created main folder: {args.save_dir}")
    else:
        print(f"Main folder already exists: {args.save_dir}")
    
    # Create the three subfolders
    subfolders = ["train", "val", "test"]
    
    for subfolder in subfolders:
        subfolder_path = os.path.join(args.save_dir, subfolder)
        if not os.path.exists(subfolder_path):
            os.makedirs(subfolder_path)
            print(f"Created subfolder: {subfolder_path}")
        else:
            print(f"Subfolder already exists: {subfolder_path}")

def create_sea_ice_variables(args):
    """
    Input: 'args defined by args_parser'
    Output: netCDF4 files

    Create input and output sea ice field 
    """
    print(args)
    create_folder_structure(args)

    #Define months used (typically only winter)
    months = range(args.months_begin, args.months_end + 1)

    #Define train years
    years_train = range(args.train_year_begin, args.train_year_end + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    years_test = range(args.test_year_begin, args.test_year_end + 1)

    
    #pattern = args.SIM_dir + f"00*_dev/NANUK1-CPL00_1h_*_icemod.nc4"
        
    #train_files = [args.SIM_dir + f"box_IA1024_NANUK12-CPL00_{y:d}{m:02d}01_{y:d}{m:02d}*_icemod.nc" for y, m in product(years_train, months)]
    mask = None
    #Create netcdf files for each stage
    nanuk = xr.open_mfdataset('/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/00*_dev/NANUK1-N1CPL00_1h_*_icemod.nc4')
    #Define months used (typically only winter)
    months = range(args.months_begin, args.months_end + 1)
    train_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_train))
    train_ds = train_ds.sel(time_counter=train_ds.time_centered.dt.month.isin(months))
    val_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_val))
    val_ds = val_ds.sel(time_counter=val_ds.time_centered.dt.month.isin(months))

    test_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_test))
    test_ds = test_ds.sel(time_counter=test_ds.time_centered.dt.month.isin(months))

    create_stage_files('train', train_ds, args, mask)
    create_stage_files('val', val_ds, args, mask)
    create_stage_files('test', test_ds, args, mask)
        

def create_stage_files( s, dataset, args, masl):
    print("Stage - "+ s)
    
    
    #Select only wanted variables
    data = dataset[args.sea_ice_variable]

    #Fill NaNs with 0
    data = data.fillna(0)
    
    #Apply the mask on all variables and all datasets
    
# Calculate padding needed
# For y: 129 -> 128, so we need to remove 1
# For x: 118 -> 128, so we need to add 10
    pad_dict = {
        'y': (0, 0),  # (pad_before, pad_after) - no padding needed, will trim later
        'x': (5, 5)   # Add 5 zeros on each side to go from 118 to 128
                }

    # First pad x dimension
    ds_padded = data.pad(pad_dict, mode='constant', constant_values=0)

# Then trim y dimension to 128
    data = ds_padded.isel(y=slice(0, 128))

    #print(np.shape(data))
    print(data)
    #Add prec coordinate
    data_with_prec = xr.Dataset(
    data_vars={
        var: (('time', 'prec', 'y', 'x'), 
              data[var].data.reshape(-1, 1, 128, 128))
        for var in args.sea_ice_variable
    },
    coords={
        "time": data.time_counter.data,
        "prec": [0],
        "x": data.x,
        "y": data.y,
        "latitude": (('y', 'x'), np.ones((128, 128))),
        "longitude": (('y', 'x'), np.ones((128, 128)))
    }
)
    data = data_with_prec

    #Compute the difference for the outputs
    data_outputs = data.shift(time=-args.timestep_difference) - data

    #Remove the first two fields filled with NaNs due to the shift
    data_outputs = data_outputs.isel(time = slice(None, -args.timestep_difference))
    data = data.isel(time = slice(None, -args.timestep_difference))

    #Compute the lead mask
    if "sit" and "sic" in args.sea_ice_variable:
        mask_lead = [data.sic<0.97][0]
        data["mask_lead"] = mask_lead

    #If necessary create folders to save data
    os.makedirs(s, exist_ok=True)

    #Split by year
    inputs_years = data.groupby('time.year')
    outputs_years = data_outputs.groupby('time.year')

    # Save each year's data to a separate NetCDF file
    for year, year_data in inputs_years:
        filename_input = args.save_dir + f'{s}/data_{year}_sea_ice_input.nc'
        year_data.isel(time = slice(args.timestep_difference, -args.timestep_difference)).to_netcdf(filename_input)
        year_data.close()
    for year, year_data in outputs_years:
        filename_output = args.save_dir + f'{s}/data_{year}_sea_ice_output.nc'
        year_data.isel(time = slice(args.timestep_difference, -args.timestep_difference)).to_netcdf(filename_output)
        year_data.close()

def get_normalisation_values_sea_ice(path_to_file = './'):
    '''
    Compute mean and standard deviation from both input and output training datasets.
    '''

    #Compute ratio between valid and invalid mask
    #mask = np.load(args.mask_path)
    #N_sum = (256*256)/np.sum(mask)

    #Open training input and output dataset
    xtrain = xr.open_mfdataset(path_to_file + "train/data_20*_sea_ice_input.nc")
    ytrain = xr.open_mfdataset(path_to_file + "train/data_20*_sea_ice_output.nc")

    #Save climatology for each variable
    climatology = xtrain.mean("time")
    climatology.to_netcdf('climatology.nc')

    #Compute mean and std. Std are multiplied by valid ratio
    mean_input = xtrain.mean(dim=["time", "x", "y", "prec"])
    std_input = xtrain.std(dim=["time", "x", "y", "prec"])#*N_sum
    mean_output = ytrain.mean(dim=["time", "x", "y", "prec"])
    std_output = ytrain.std(dim=["time", "x", "y", "prec"])#*N_sum

    #Save normalization values
    mean_input.to_netcdf(args.save_dir+'sea_ice_mean_input.nc')
    mean_output.to_netcdf(args.save_dir+'sea_ice_mean_output.nc')
    std_input.to_netcdf(args.save_dir+'sea_ice_std_input.nc')
    std_output.to_netcdf(args.save_dir+'sea_ice_std_output.nc')

def apply_normalisation_sea_ice():
    
    N_sum = (128*128)
    i_o = ["input", "output"]
    for i in i_o:
        mean = xr.open_dataset(args.save_dir+"sea_ice_mean_"+i+".nc")
        std = xr.open_dataset(args.save_dir+"sea_ice_std_"+i+".nc")
        stage = ['train', 'val', 'test']
        for s in stage: 
            x = xr.open_mfdataset(args.save_dir+f'{s}/data_20*_sea_ice_'+ i +".nc")
            x = (x - mean) / std
            x_years = x.groupby('time.year')
            for year, year_data in x_years:
                filename_input = args.save_dir+f'{s}/data_{year}_sea_ice_'+i+'_normalized.nc'
                year_data.to_netcdf(filename_input)
                year_data.close()
        

# Create sea ice dataset

In [18]:
create_sea_ice_variables(args)


Namespace(SIM_dir='/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/', atmo_variable=['t2m', 'mtpr', 'd2m', 'msdlwrf', 'msdswrf'], mask_path='mask_SIL_regions.npy', months_begin=1, months_end=12, ocean_forcings=['vozocrtx', 'vomecrty'], ocean_variable=['zos', 'tos', 'sos'], path_to_atmo_forcings='/summer/sasip/model-forcings/atmo_forcing/ERA5_full/', save_dir='/summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/', sea_ice_variable=['sivolu', 'siconc', 'u_ice', 'v_ice', 'snvolu'], test_year_begin=2019, test_year_end=2020, timestep_difference=6, train_year_begin=2010, train_year_end=2017, val_year_begin=2018, val_year_end=2018, x_max=0, x_min=0, y_max=0, y_min=0)
Created main folder: /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/
Created subfolder: /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/train
Created subfolder: /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/val
Created subfolder: /summer/meom/workdir

/home/ducharlo/.conda/envs/nextsim_surrogate/lib/python3.8/site-packages/xarray/core/indexing.py:1379: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/home/ducharlo/.conda/envs/nextsim_surrogate/lib/python3.8/site-packages/xarray/core/indexing.py:1379: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  ret

Stage - train
<xarray.Dataset>
Dimensions:        (time_counter: 70128, y: 128, x: 128)
Coordinates:
    nav_lat        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    nav_lon        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    time_centered  (time_counter) datetime64[ns] dask.array<chunksize=(8760,), meta=np.ndarray>
  * time_counter   (time_counter) datetime64[ns] 2010-01-01T00:30:00 ... 2017...
Dimensions without coordinates: y, x
Data variables:
    sivolu         (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    siconc         (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    u_ice          (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    v_ice          (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    snvolu         (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndar

/tmp/ipykernel_2111442/2572283162.py:1: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  create_sea_ice_variables(args)


Stage - test
<xarray.Dataset>
Dimensions:        (time_counter: 17544, y: 128, x: 128)
Coordinates:
    nav_lat        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    nav_lon        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    time_centered  (time_counter) datetime64[ns] dask.array<chunksize=(8760,), meta=np.ndarray>
  * time_counter   (time_counter) datetime64[ns] 2019-01-01T00:30:00 ... 2020...
Dimensions without coordinates: y, x
Data variables:
    sivolu         (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    siconc         (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    u_ice          (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    v_ice          (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    snvolu         (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarr

/tmp/ipykernel_2111442/2572283162.py:1: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  create_sea_ice_variables(args)


# Compute normalisation values

In [19]:
get_normalisation_values_sea_ice(args.save_dir)

# Apply normalisation values

In [20]:
apply_normalisation_sea_ice()

In [21]:
def create_ocean_variables(args):
    """
    Input: 'args defined by args_parser'
    Output: netCDF4 files

    Create input and output sea ice field 
    """
    print(args)
    create_folder_structure(args)

    #Define months used (typically only winter)
    months = range(args.months_begin, args.months_end + 1)

    #Define train years
    years_train = range(args.train_year_begin, args.train_year_end + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    years_test = range(args.test_year_begin, args.test_year_end + 1)

    
    #pattern = args.SIM_dir + f"00*_dev/NANUK1-CPL00_1h_*_icemod.nc4"
        
    #train_files = [args.SIM_dir + f"box_IA1024_NANUK12-CPL00_{y:d}{m:02d}01_{y:d}{m:02d}*_icemod.nc" for y, m in product(years_train, months)]
    mask = None
    #Create netcdf files for each stage
    nanuk = xr.open_mfdataset('/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/00*_dev/NANUK1-N1CPL00_1h_*_icemod.nc4')

    train_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_train))
    train_ds = train_ds.sel(time_counter=train_ds.time_centered.dt.month.isin(months))
    val_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_val))
    val_ds = val_ds.sel(time_counter=val_ds.time_centered.dt.month.isin(months))

    test_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_test))
    test_ds = test_ds.sel(time_counter=test_ds.time_centered.dt.month.isin(months))
    create_stage_files('train', train_ds, args, mask)
    create_stage_files('val', val_ds, args, mask)
    create_stage_files('test', test_ds, args, mask)
        

def create_stage_files( s, dataset, args, masl):
    print("Stage - "+ s)
    
    
    #Select only wanted variables
    data = dataset[args.ocean_variable]

    #Fill NaNs with 0
    data = data.fillna(0)
    
    #Apply the mask on all variables and all datasets
    
# Calculate padding needed
# For y: 129 -> 128, so we need to remove 1
# For x: 118 -> 128, so we need to add 10
    pad_dict = {
        'y': (0, 0),  # (pad_before, pad_after) - no padding needed, will trim later
        'x': (5, 5)   # Add 5 zeros on each side to go from 118 to 128
                }

    # First pad x dimension
    ds_padded = data.pad(pad_dict, mode='constant', constant_values=0)

# Then trim y dimension to 128
    data = ds_padded.isel(y=slice(0, 128))

    #print(np.shape(data))
    print(data)
    #Add prec coordinate
    data_with_prec = xr.Dataset(
    data_vars={
        var: (('time', 'prec', 'y', 'x'), 
              data[var].data.reshape(-1, 1, 128, 128))
        for var in args.ocean_variable
    },
    coords={
        "time": data.time_counter.data,
        "prec": [0],
        "x": data.x,
        "y": data.y,
        "latitude": (('y', 'x'), np.ones((128, 128))),
        "longitude": (('y', 'x'), np.ones((128, 128)))
    }
)
    data = data_with_prec

    #Compute the difference for the outputs
    #data_outputs = data.shift(time=-args.timestep_difference) - data

    #Remove the first two fields filled with NaNs due to the shift
    #data_outputs = data_outputs.isel(time = slice(None, -args.timestep_difference))
    data = data.isel(time = slice(None, -args.timestep_difference))

    #Compute the lead mask
    #if "sit" and "sic" in args.sea_ice_variable:
     #   mask_lead = [data.sic<0.97][0]
      #  data["mask_lead"] = mask_lead

    #If necessary create folders to save data
    os.makedirs(s, exist_ok=True)

    #Split by year
    inputs_years = data.groupby('time.year')
    #outputs_years = data_outputs.groupby('time.year')

    # Save each year's data to a separate NetCDF file
    for year, year_data in inputs_years:
        filename_input = args.save_dir + f'{s}/data_{year}_ocean_input.nc'
        year_data.isel(time = slice(args.timestep_difference, -args.timestep_difference)).to_netcdf(filename_input)
        year_data.close()
    #for year, year_data in outputs_years:
     #   filename_output = args.save_dir + f'{s}/data_{year}_sea_ice_output.nc'
      #  year_data.isel(time = slice(args.timestep_difference, -args.timestep_difference)).to_netcdf(filename_output)
       # year_data.close()

def get_normalisation_values_ocean(path_to_file = './'):
    '''
    Compute mean and standard deviation from both input and output training datasets.
    '''

    #Compute ratio between valid and invalid mask
    #mask = np.load(args.mask_path)
    #N_sum = (256*256)/np.sum(mask)

    #Open training input and output dataset
    xtrain = xr.open_mfdataset(path_to_file + "train/data_20*_ocean_input.nc")

    #Save climatology for each variable
    climatology = xtrain.mean("time")
    climatology.to_netcdf('climatology.nc')

    #Compute mean and std. Std are multiplied by valid ratio
    mean_input = xtrain.mean(dim=["time", "x", "y", "prec"])
    std_input = xtrain.std(dim=["time", "x", "y", "prec"])#*N_sum
    
    #Save normalization values
    mean_input.to_netcdf(args.save_dir+'ocean_mean_input.nc')
    std_input.to_netcdf(args.save_dir+'ocean_std_input.nc')

def apply_normalisation_ocean():
    
    N_sum = (128*128)
    i_o = ["input"]
    for i in i_o:
        mean = xr.open_dataset(args.save_dir+"ocean_mean_"+i+".nc")
        std = xr.open_dataset(args.save_dir+"ocean_std_"+i+".nc")
        stage = ['train', 'val', 'test']
        for s in stage: 
            x = xr.open_mfdataset(args.save_dir+f'{s}/data_20*_ocean_'+ i +".nc")
            x = (x - mean) / std
            x_years = x.groupby('time.year')
            for year, year_data in x_years:
                filename_input = args.save_dir+f'{s}/data_{year}_ocean_'+i+'_normalized.nc'
                year_data.to_netcdf(filename_input)
                year_data.close()
        

In [22]:
create_ocean_variables(args)
get_normalisation_values_ocean(args.save_dir)
apply_normalisation_ocean()

Namespace(SIM_dir='/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/', atmo_variable=['t2m', 'mtpr', 'd2m', 'msdlwrf', 'msdswrf'], mask_path='mask_SIL_regions.npy', months_begin=1, months_end=12, ocean_forcings=['vozocrtx', 'vomecrty'], ocean_variable=['zos', 'tos', 'sos'], path_to_atmo_forcings='/summer/sasip/model-forcings/atmo_forcing/ERA5_full/', save_dir='/summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/', sea_ice_variable=['sivolu', 'siconc', 'u_ice', 'v_ice', 'snvolu'], test_year_begin=2019, test_year_end=2020, timestep_difference=6, train_year_begin=2010, train_year_end=2017, val_year_begin=2018, val_year_end=2018, x_max=0, x_min=0, y_max=0, y_min=0)
Main folder already exists: /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/
Subfolder already exists: /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/train
Subfolder already exists: /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/val
Subfolder already 

/home/ducharlo/.conda/envs/nextsim_surrogate/lib/python3.8/site-packages/xarray/core/indexing.py:1379: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/home/ducharlo/.conda/envs/nextsim_surrogate/lib/python3.8/site-packages/xarray/core/indexing.py:1379: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  ret

Stage - train
<xarray.Dataset>
Dimensions:        (time_counter: 70128, y: 128, x: 128)
Coordinates:
    nav_lat        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    nav_lon        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    time_centered  (time_counter) datetime64[ns] dask.array<chunksize=(8760,), meta=np.ndarray>
  * time_counter   (time_counter) datetime64[ns] 2010-01-01T00:30:00 ... 2017...
Dimensions without coordinates: y, x
Data variables:
    zos            (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    tos            (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    sos            (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
Attributes:
    name:         /workdir/meom/brodeau/NANUK1/NANUK1-N1CPL00-S/00000001-0000...
    description:  ice variables
    title:        ice variables
    Conventions:  CF-1.6
    timeStamp:   

/tmp/ipykernel_2111442/4213567161.py:1: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  create_ocean_variables(args)


Stage - test
<xarray.Dataset>
Dimensions:        (time_counter: 17544, y: 128, x: 128)
Coordinates:
    nav_lat        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    nav_lon        (y, x) float32 dask.array<chunksize=(128, 5), meta=np.ndarray>
    time_centered  (time_counter) datetime64[ns] dask.array<chunksize=(8760,), meta=np.ndarray>
  * time_counter   (time_counter) datetime64[ns] 2019-01-01T00:30:00 ... 2020...
Dimensions without coordinates: y, x
Data variables:
    zos            (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    tos            (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
    sos            (time_counter, y, x) float32 dask.array<chunksize=(8760, 128, 5), meta=np.ndarray>
Attributes:
    name:         /workdir/meom/brodeau/NANUK1/NANUK1-N1CPL00-S/00000001-0000...
    description:  ice variables
    title:        ice variables
    Conventions:  CF-1.6
    timeStamp:    

/tmp/ipykernel_2111442/4213567161.py:1: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  create_ocean_variables(args)


In [24]:
create_ocean_variables_forcings(args)
get_normalisation_values_ocean_forcings(args.save_dir)
apply_normalisation_ocean_forcings()

Namespace(SIM_dir='/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/', atmo_variable=['t2m', 'mtpr', 'd2m', 'msdlwrf', 'msdswrf'], mask_path='mask_SIL_regions.npy', months_begin=1, months_end=12, ocean_forcings=['vozocrtx', 'vomecrty'], ocean_variable=['zos', 'tos', 'sos'], path_to_atmo_forcings='/summer/sasip/model-forcings/atmo_forcing/ERA5_full/', save_dir='/summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/', sea_ice_variable=['sivolu', 'siconc', 'u_ice', 'v_ice', 'snvolu'], test_year_begin=2019, test_year_end=2020, timestep_difference=6, train_year_begin=2010, train_year_end=2017, val_year_begin=2018, val_year_end=2018, x_max=0, x_min=0, y_max=0, y_min=0)


/home/ducharlo/.conda/envs/nextsim_surrogate/lib/python3.8/site-packages/xarray/core/indexing.py:1379: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/home/ducharlo/.conda/envs/nextsim_surrogate/lib/python3.8/site-packages/xarray/core/indexing.py:1379: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  ret

Stage - train
<xarray.Dataset>
Dimensions:         (time_counter: 70128, y_grid_U: 129, x_grid_U: 118,
                     y_grid_V: 129, x_grid_V: 118)
Coordinates:
    nav_lat_grid_U  (y_grid_U, x_grid_U) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    nav_lon_grid_U  (y_grid_U, x_grid_U) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    deptht30        float32 22.76
    nav_lat_grid_V  (y_grid_V, x_grid_V) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    nav_lon_grid_V  (y_grid_V, x_grid_V) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    time_centered   (time_counter) datetime64[ns] dask.array<chunksize=(8760,), meta=np.ndarray>
  * time_counter    (time_counter) datetime64[ns] 2010-01-01T00:30:00 ... 201...
Dimensions without coordinates: y_grid_U, x_grid_U, y_grid_V, x_grid_V
Data variables:
    vozocrtx        (time_counter, y_grid_U, x_grid_U) float32 dask.array<chunksize=(8760, 129, 118), meta=np.ndarray>
    vomecrty       

/tmp/ipykernel_2111442/1698711746.py:1: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  create_ocean_variables_forcings(args)


Stage - test
<xarray.Dataset>
Dimensions:         (time_counter: 17544, y_grid_U: 129, x_grid_U: 118,
                     y_grid_V: 129, x_grid_V: 118)
Coordinates:
    nav_lat_grid_U  (y_grid_U, x_grid_U) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    nav_lon_grid_U  (y_grid_U, x_grid_U) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    deptht30        float32 22.76
    nav_lat_grid_V  (y_grid_V, x_grid_V) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    nav_lon_grid_V  (y_grid_V, x_grid_V) float32 dask.array<chunksize=(129, 118), meta=np.ndarray>
    time_centered   (time_counter) datetime64[ns] dask.array<chunksize=(8760,), meta=np.ndarray>
  * time_counter    (time_counter) datetime64[ns] 2019-01-01T00:30:00 ... 202...
Dimensions without coordinates: y_grid_U, x_grid_U, y_grid_V, x_grid_V
Data variables:
    vozocrtx        (time_counter, y_grid_U, x_grid_U) float32 dask.array<chunksize=(8760, 129, 118), meta=np.ndarray>
    vomecrty        

/tmp/ipykernel_2111442/1698711746.py:1: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  create_ocean_variables_forcings(args)


In [23]:
def create_ocean_variables_forcings(args):
    """
    Input: 'args defined by args_parser'
    Output: netCDF4 files

    Create input and output sea ice field 
    """
    print(args)
    
    #Define months used (typically only winter)
    months = range(args.months_begin, args.months_end + 1)

    #Define train years
    years_train = range(args.train_year_begin, args.train_year_end + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    years_test = range(args.test_year_begin, args.test_year_end + 1)

    
    #pattern = args.SIM_dir + f"00*_dev/NANUK1-CPL00_1h_*_icemod.nc4"
        
    #train_files = [args.SIM_dir + f"box_IA1024_NANUK12-CPL00_{y:d}{m:02d}01_{y:d}{m:02d}*_icemod.nc" for y, m in product(years_train, months)]
    mask = None
    #Create netcdf files for each stage
    nanuk = xr.open_mfdataset('/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-N1CPL00-S/00*_dev/NANUK1-N1CPL00_1h_*_ovel30.nc4')

    train_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_train))
    val_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_val))
    test_ds = nanuk.sel(time_counter=nanuk.time_centered.dt.year.isin(years_test))
    create_ocean_forcings('train', train_ds, args, mask)
    create_ocean_forcings('val', val_ds, args, mask)
    create_ocean_forcings('test', test_ds, args, mask)
        

def create_ocean_forcings( s, dataset, args, mask):
    print("Stage - "+ s)
    
    
    #Select only wanted variables
    data = dataset[args.ocean_forcings].isel(deptht30=0)
    data
    #Fill NaNs with 0
    data = data.fillna(0)
    
    
# Calculate padding needed
# For y: 129 -> 128, so we need to remove 1
# For x: 118 -> 128, so we need to add 10
    pad_dict = {
        'y_grid_U': (0, 0),  # (pad_before, pad_after) - no padding needed, will trim later
        'x_grid_U': (5, 5),   # Add 5 zeros on each side to go from 118 to 128
        'y_grid_V': (0, 0),  # (pad_before, pad_after) - no padding needed, will trim later
        'x_grid_V': (5, 5)   # Add 5 zeros on each side to go from 118 to 128
                }
    print(data)
    # First pad x dimension
    ds_padded = data.pad(pad_dict, mode='constant', constant_values=0)
    print(ds_padded)
# Then trim y dimension to 128
    data = ds_padded.isel(y_grid_U=slice(0, 128), y_grid_V=slice(0, 128))
    print(data)
    #print(np.shape(data))
    print(data)
    #Add prec coordinate
    data_with_prec = xr.Dataset(
    data_vars={
        var: (('time', 'prec', 'y', 'x'), 
              data[var].data.reshape(-1, 1, 128, 128))
        for var in args.ocean_forcings
    },
    coords={
        "time": data.time_counter.data,
        "prec": [0],
        "x": np.arange(128),
        "y": np.arange(128),
        "latitude": (('y', 'x'), np.ones((128, 128))),
        "longitude": (('y', 'x'), np.ones((128, 128)))
    }
)
    data = data_with_prec

    #Compute the lead mask
    #if "sit" and "sic" in args.sea_ice_variable:
     #   mask_lead = [data.sic<0.97][0]
      #  data["mask_lead"] = mask_lead

    #If necessary create folders to save data
    os.makedirs(s, exist_ok=True)

    #Split by year
    inputs_years = data.groupby('time.year')

    # Save each year's data to a separate NetCDF file
    for year, year_data in inputs_years:
        filename_input = args.save_dir + f'{s}/data_{year}_ocean_forcings_input.nc'
        year_data.isel(time = slice(args.timestep_difference, -args.timestep_difference)).to_netcdf(filename_input)
        year_data.close()
    

def get_normalisation_values_ocean_forcings(path_to_file = './'):
    '''
    Compute mean and standard deviation from both input and output training datasets.
    '''

    #Compute ratio between valid and invalid mask
    #mask = np.load(args.mask_path)
    #N_sum = (256*256)/np.sum(mask)

    #Open training input and output dataset
    xtrain = xr.open_mfdataset(path_to_file + "train/data_20*_ocean_forcings_input.nc")

    #Save climatology for each variable
    climatology = xtrain.mean("time")
    climatology.to_netcdf('climatology.nc')

    #Compute mean and std. Std are multiplied by valid ratio
    mean_input = xtrain.mean(dim=["time", "x", "y", "prec"])
    std_input = xtrain.std(dim=["time", "x", "y", "prec"])#*N_sum
   
    #Save normalization values
    mean_input.to_netcdf(args.save_dir+'ocean_forcings_under_mean_input.nc')
    std_input.to_netcdf(args.save_dir+'ocean_forcings_under_std_input.nc')


def apply_normalisation_ocean_forcings():
    
    N_sum = (128*128)
    i_o = ["input"]
    for i in i_o:
        mean = xr.open_dataset(args.save_dir+"ocean_forcings_under_mean_"+i+".nc")
        std = xr.open_dataset(args.save_dir+"ocean_forcings_under_std_"+i+".nc")
        stage = ['train', 'val', 'test']
        for s in stage: 
            x = xr.open_mfdataset(args.save_dir+f'{s}/data_20*_ocean_forcings_'+ i +".nc")
            x = (x - mean) / std
            x_years = x.groupby('time.year')
            for year, year_data in x_years:
                filename_input = args.save_dir+f'{s}/data_{year}_ocean_forcings_'+i+'_normalized.nc'
                year_data.to_netcdf(filename_input)
                year_data.close()
        

In [3]:


def create_atmo_winds(args):
    #dask.config.set({'array.slicing.split_large_chunks': False})
    #Load mask for Central Arctic
    

    #Define months used (typically only winter)
    months = range(args.months_begin, args.months_end + 1)

    #Define train years
    years_train = range(args.train_year_begin, args.train_year_end + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    years_test = range(args.test_year_begin, args.test_year_end + 1)

    
    stage = ['train', 'val', 'test']
    
    #Create netcdf files for each stage
    for y in years_train:
        print(y)
        create_stage_files_atmo_winds('train', y, args, months)
    for y in years_val:
        print(y)
        create_stage_files_atmo_winds('val', y, args, months)
    for y in years_test:
        print(y)
        create_stage_files_atmo_winds('test', y, args, months)
    
def create_stage_files_atmo_winds( s, y, args, months):

    #Get neXtSIM grid latitude and longitude
    
    #Get ERA5 grid latitude and longitude
    ERA5 = xr.open_dataset('/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/ERA5_wspeed10m_NANUK1/u10_ERA5-NANUK1_gridT_y2010.nc')        
    forcings_files = [f'/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/ERA5_wspeed10m_NANUK1/{v}_ERA5-NANUK1_gridT_y{y:d}.nc' for v in ['u10','v10']]
    # Process files in smaller batches
    forcings = []

    # Open dataset for this batch
    batch_ds = xr.open_mfdataset(
                forcings_files,
                engine="netcdf4"
                )
    forcings.append(batch_ds)
    months = range(args.months_begin, args.months_end + 1)
    forcing = xr.merge(forcings)
    forcing = forcing.isel(time_counter=forcing.time_counter.dt.month.isin(months))
    interpolated_vars = {}

    #Shape of interpolated images
    N_x = 118
    N_y = 129
    
    # Second interpolated dataset creation with prec dimension
    interpolated_dataset = xr.Dataset(
        {var: (('time_counter', 'prec', 'y', 'x'), 
                np.reshape(forcing[var].data, (-1, 1, N_y, N_x))
              ) 
         for var in ['u10','v10']},
        coords={
            'time_counter': forcing.time_counter,
            'prec': np.arange(1),
            'latitude': (["y", "x"], forcing.nav_lat.data),
            'longitude': (["y", "x"], forcing.nav_lon.data)
        }
    )
    #forcings_interpolated = interpolate_all_variables(forcing, lat_source, inds, nextsim)

    #Select data with the same dimension as neXtSIM
    #forcings_interpolated = forcing.isel(y=slice(args.y_min, args.y_max), x=slice(args.x_min, args.x_max))
    #forcings_interpolated = interpolated_dataset.isel(time = slice(args.timestep_difference, -args.timestep_difference))
    forcings_interpolated = interpolated_dataset.isel(time_counter = slice(args.timestep_difference, -args.timestep_difference))
    #forcings_interpolated = interpolated_dataset.isel(time = slice(args.timestep_difference, -args.timestep_difference))

        #Save file
    #forcings_interpolated.drop_dims('time_counter')
    filename_input = args.save_dir + f'{s}/data_{y}_winds_input.nc'
    forcings_interpolated.to_netcdf(filename_input)
    forcings_interpolated.close()
def get_normalisation_values_atmo_winds(path_to_file = './'):
    '''
    Compute mean and standard deviation from both input and output training datasets for atmospheric forcings.
    '''
    #Open training input and output dataset
    xtrain = xr.open_mfdataset(path_to_file + "train/*_winds_input_reshape.nc")
    #xtrain.drop_dims('time_counter')
    #Compute mean and std
    mean_input = xtrain.mean(dim=["time_counter", "x", "y"])
    std_input = xtrain.std(dim=["time_counter", "x", "y"])

    #Save normalization values
    mean_input.to_netcdf(args.save_dir +'winds_mean_input.nc')
    std_input.to_netcdf(args.save_dir +'winds_std_input.nc')
    

def apply_normalisation_atmo_winds():
    i_o = ["input"]
    for i in i_o:
        mean = xr.open_dataset(args.save_dir +"winds_mean_"+i+".nc")
        std = xr.open_dataset(args.save_dir +"winds_std_"+i+".nc")
        stage = ['train', 'val', 'test']
        for s in stage: 
            x = xr.open_mfdataset(args.save_dir +s + "/data_*_winds_"+ i +"_reshape.nc")
            x = (x - mean) / std
            x_years = x.groupby('time_counter.year')
            for year, year_data in x_years:
                print(year)
                filename_input = args.save_dir +f'{s}/data_{year}_winds_'+i+'_normalized.nc'
                year_data.to_netcdf(filename_input)
                year_data.close()

def create_atmo_reshape_winds(args):
    #dask.config.set({'array.slicing.split_large_chunks': False})
    #Load mask for Central Arctic
    

    #Define months used (typically only winter)
    months = range(args.months_begin, args.months_end + 1)

    #Define train years
    years_train = range(args.train_year_begin, args.train_year_end + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    years_test = range(args.test_year_begin, args.test_year_end + 1)

    
    stage = ['train', 'val', 'test']
    
    #Create netcdf files for each stage
    for y in years_train:
        reshape_stage_files_atmo_winds('train', y, args, months)
    for y in years_val:
        reshape_stage_files_atmo_winds('val', y, args, months)
    for y in years_test:
        reshape_stage_files_atmo_winds('test', y, args, months)

def reshape_stage_files_atmo_winds( s, y, args, months):

    #Get neXtSIM grid latitude and longitude
    
    forcings_files = args.save_dir + f"{s}/data_{y:d}_winds_input.nc"
        # Process files in smaller batches
    

        # Open dataset for this batch
    forcing = xr.open_dataset(
                forcings_files,
                engine="netcdf4"
                )
    
    print(forcing)
        #Create a boolean mask for the specific hours you want
        #hour_mask = (hours == 3) | (hours == 6) |(hours == 9) | (hours == 12) |(hours == 15) |(hours == 18) | (hours == 21)| (hours == 24)
    
    pad_dict = {
        'y': (0, 0),  # (pad_before, pad_after) - no padding needed, will trim later
        'x': (5, 5)   # Add 5 zeros on each side to go from 118 to 128
                }

    # First pad x dimension
    forcings_reshape = forcing.pad(pad_dict, mode='constant', constant_values=0)

# Then trim y dimension to 128
    forcings_reshape = forcings_reshape.isel(y=slice(0, 128))

        #Save file
        
    filename_input = args.save_dir + f'{s}/data_{y}_winds_input_reshape.nc'
    
    forcings_reshape.to_netcdf(filename_input)
    forcings_reshape.close()

In [26]:
create_atmo_winds(args)
create_atmo_reshape_winds(args)
get_normalisation_values_atmo_winds(args.save_dir)
apply_normalisation_atmo_winds()

2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
<xarray.Dataset>
Dimensions:       (time_counter: 8748, prec: 1, y: 129, x: 118)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 2010-01-01T06:00:00 ... 2010-...
  * prec          (prec) int64 0
    latitude      (y, x) float64 ...
    longitude     (y, x) float64 ...
Dimensions without coordinates: y, x
Data variables:
    u10           (time_counter, prec, y, x) float32 ...
    v10           (time_counter, prec, y, x) float32 ...
<xarray.Dataset>
Dimensions:       (time_counter: 8748, prec: 1, y: 129, x: 118)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 2011-01-01T06:00:00 ... 2011-...
  * prec          (prec) int64 0
    latitude      (y, x) float64 ...
    longitude     (y, x) float64 ...
Dimensions without coordinates: y, x
Data variables:
    u10           (time_counter, prec, y, x) float32 ...
    v10           (time_counter, prec, y, x) float32 ...
<xarray.Dataset>
Dimensions:       (time_counte

In [4]:
def interpolate_all_variables(dataset, target_shape, inds, nextsim):
    """
    Interpolate all variables in an xarray dataset efficiently.
    
    Parameters:
    - dataset: Input xarray Dataset
    - target_shape: Target shape for reshaping
    - inds: Indices to select from the flattened array
    
    Returns:
    - xarray Dataset with interpolated variables
    """
    # Dictionary to store interpolated variables
    interpolated_vars = {}

    #Shape of interpolated images
    N_x = np.shape(target_shape)[0]
    N_y = np.shape(target_shape)[1]
    
    # Iterate through all data variables in the dataset
    for var_name in dataset.data_vars:
        print(var_name)
        # Select the variable and flatten with indices
        var = dataset[var_name].values.reshape(dataset[var_name].shape[0], -1)[:, inds]
        # Reshape to target shape
        var = var.reshape(var.shape[0], *target_shape.shape) 
        # Store the interpolated variable
        interpolated_vars[var_name] = var

        
    # First interpolated dataset creation
   # interpolated_dataset = xr.Dataset(
    #    {var: (('time', 'y', 'x'), interpolated_vars[var]) 
     #    for var in interpolated_vars},
      #  coords={
       #     'time': dataset.time,
        #    'latitude': (["y", "x"], nextsim.latitude.data),
         #   'longitude': (["y", "x"], nextsim.longitude.data)
        #}
    #)

    # Second interpolated dataset creation with prec dimension
    interpolated_dataset = xr.Dataset(
        {var: (('time', 'prec', 'y', 'x'), 
                np.reshape(interpolated_vars[var].data, (-1, 1, N_x, N_y))
              ) 
         for var in interpolated_vars},
        coords={
            'time': dataset.time,
            'prec': np.arange(1),
            'latitude': (["y", "x"], nextsim.nav_lat.data),
            'longitude': (["y", "x"], nextsim.nav_lon.data)
        }
    )
    return interpolated_dataset

def lon_lat_to_cartesian(lon, lat):
        # WGS 84 reference coordinate system parameters
    A = 6378.137  # major axis [km]
    E2 = 6.69437999014e-3  # eccentricity squared

    lon_rad = np.radians(lon)
    lat_rad = np.radians(lat)
    # convert to cartesian coordinates
    r_n = A / (np.sqrt(1 - E2 * (np.sin(lat_rad) ** 2)))
    x = r_n * np.cos(lat_rad) * np.cos(lon_rad)
    y = r_n * np.cos(lat_rad) * np.sin(lon_rad)
    z = r_n * (1 - E2) * np.sin(lat_rad)
    return x, y, z

def create_atmo_variables(args):
    #dask.config.set({'array.slicing.split_large_chunks': False})
    #Load mask for Central Arctic
    

    #Define months used (typically only winter)
    months = range(args.months_begin, args.months_end + 1)

    #Define train years
    #years_train = range(2015, args.train_year_end + 1)
    years_train = range(args.train_year_begin, args.train_year_end + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    years_test = range(args.test_year_begin, args.test_year_end + 1)

    
    stage = ['train', 'val', 'test']
    
    #Create netcdf files for each stage
    #for y in years_train:
     #   print(y)
      #  create_stage_files_atmo('train', y, args, months)
    for y in years_val:
        print(y)
        create_stage_files_atmo('val', y, args, months)
    for y in years_test:
        print(y)
        create_stage_files_atmo('test', y, args, months)
    
def create_stage_files_atmo( s, y, args, months):

    #Get neXtSIM grid latitude and longitude
    nextsim = xr.open_mfdataset('/summer/meom/MEOM-OPENDAP/simus_nanuq/NANUK1/NANUK1-CPL00-S/00*_dev/NANUK1-CPL00_1h_*_icemod.nc4')
    lat_source = nextsim.variables["nav_lat"]
    lon_source = nextsim.variables["nav_lon"]
    #Get ERA5 grid latitude and longitude
    months = range(args.months_begin, args.months_end + 1)
    ERA5 = xr.open_dataset(args.path_to_atmo_forcings + f"ERA5_hourly_t2m_y{y:d}.nc")
    lat_target = ERA5.lat
    lon_target = ERA5.lon
    lon_target2d, lat_target2d = np.meshgrid(lon_target, lat_target)
    #Convert neXtSIM latlon to cartesian 
    xt, yt, zt = lon_lat_to_cartesian(
                lon_source.values.flatten(), lat_source.values.flatten()
                   )
    #Convert ERA5 latlon to cartesian 
    xs, ys, zs = lon_lat_to_cartesian(lon_target2d.flatten(), lat_target2d.flatten())

    #Compute Nearest Neighbors
    tree = cKDTree(np.column_stack((xs, ys, zs)))
    d, inds = tree.query(np.column_stack((xt, yt, zt)), k=1)
    
    forcings_files = [args.path_to_atmo_forcings + f"ERA5_hourly_{v}_y{y:d}.nc" for v in args.atmo_variable]
        # Process files in smaller batches
    forcings = []
    print("Open forcings")
        # Open dataset for this batch
    batch_ds = xr.open_mfdataset(
                forcings_files,
                engine="netcdf4"
                )
    forcings.append(batch_ds)
    print("Merge forcings")
    forcing = xr.merge(forcings)
        #Create a boolean mask for the specific hours you want
        #hour_mask = (hours == 3) | (hours == 6) |(hours == 9) | (hours == 12) |(hours == 15) |(hours == 18) | (hours == 21)| (hours == 24)
    
    print('select months')
    forcing = forcing.isel(time=forcing.time.dt.month.isin(months))
    print('do the interpolation')
    forcings_interpolated = interpolate_all_variables(forcing, lat_source, inds, nextsim)

    #Select data with the same dimension as neXtSIM
    #forcings_interpolated = forcings_interpolated.isel(y=slice(args.y_min, args.y_max), x=slice(args.x_min, args.x_max))
    forcings_interpolated = forcings_interpolated.isel(time = slice(args.timestep_difference, -args.timestep_difference))

        #Save file
        
    filename_input = args.save_dir + f'{s}/data_{y}_atmo_input.nc'
    
    forcings_interpolated.to_netcdf(filename_input)
    forcings_interpolated.close()
def get_normalisation_values_atmo(path_to_file = './'):
    '''
    Compute mean and standard deviation from both input and output training datasets for atmospheric forcings.
    '''
    #Open training input and output dataset
    xtrain = xr.open_mfdataset(path_to_file + "train/*_atmo_input_reshape.nc")

    #Compute mean and std
    mean_input = xtrain.mean(dim=["time", "x", "y"])
    std_input = xtrain.std(dim=["time", "x", "y"])

    #Save normalization values
    mean_input.to_netcdf(args.save_dir +'atmo_mean_input.nc')
    std_input.to_netcdf(args.save_dir +'atmo_std_input.nc')
    

def apply_normalisation_atmo():
    i_o = ["input"]
    for i in i_o:
        mean = xr.open_dataset(args.save_dir +"atmo_mean_"+i+".nc")
        std = xr.open_dataset(args.save_dir +"atmo_std_"+i+".nc")
        stage = ['train', 'val', 'test']
        for s in stage: 
            x = xr.open_mfdataset(args.save_dir +s + "/data_*_atmo_"+ i +"_reshape.nc")
            x = (x - mean) / std
            x_years = x.groupby('time.year')
            for year, year_data in x_years:
                print(year)
                filename_input = args.save_dir +f'{s}/data_{year}_atmo_'+i+'_normalized.nc'
                year_data.to_netcdf(filename_input)
                year_data.close()

def create_atmo_reshape(args):
    #dask.config.set({'array.slicing.split_large_chunks': False})
    #Load mask for Central Arctic
    

    #Define months uased (typically only winter)
    months = range(args.months_begin, args.months_end + 1)

    #Define train years
    years_train = range(args.train_year_begin, args.train_year_end + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    years_test = range(args.test_year_begin, args.test_year_end + 1)

    
    stage = ['train', 'val', 'test']
    
    #Create netcdf files for each stage
    for y in years_train:
        print(y)
        reshape_stage_files_atmo('train', y, args, months)
    for y in years_val:
        print(y)
        reshape_stage_files_atmo('val', y, args, months)
    for y in years_test:
        print(y)
        reshape_stage_files_atmo('test', y, args, months)

def reshape_stage_files_atmo( s, y, args, months):

    #Get neXtSIM grid latitude and longitude
    
    forcings_files = args.save_dir + f"{s}/data_{y:d}_atmo_input.nc"
        # Process files in smaller batches
    

        # Open dataset for this batch
    forcing = xr.open_dataset(
                forcings_files,
                engine="netcdf4"
                )
    
        #Create a boolean mask for the specific hours you want
        #hour_mask = (hours == 3) | (hours == 6) |(hours == 9) | (hours == 12) |(hours == 15) |(hours == 18) | (hours == 21)| (hours == 24)
    
    pad_dict = {
        'y': (0, 0),  # (pad_before, pad_after) - no padding needed, will trim later
        'x': (5, 5)   # Add 5 zeros on each side to go from 118 to 128
                }

    # First pad x dimension
    forcings_reshape = forcing.pad(pad_dict, mode='constant', constant_values=0)

# Then trim y dimension to 128
    forcings_reshape = forcings_reshape.isel(y=slice(0, 128))

        #Save file
        
    filename_input = args.save_dir + f'{s}/data_{y}_atmo_input_reshape.nc'
    
    forcings_reshape.to_netcdf(filename_input)
    forcings_reshape.close()

In [5]:
create_atmo_variables(args)
create_atmo_reshape(args)
get_normalisation_values_atmo(args.save_dir)
apply_normalisation_atmo()

2018
Open forcings
Merge forcings
select months
do the interpolation
var167
var168
var35
var36
var55
2019
Open forcings
Merge forcings
select months
do the interpolation
var167
var168
var35
var36
var55
2020
Open forcings
Merge forcings
select months
do the interpolation
var167
var168
var35
var36
var55
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020


In [14]:
def open_winds_fields(s, args):
    x = xr.open_mfdataset(args.save_dir + s + "/data_20*_winds_input_normalized.nc")
    return x
    
def open_sea_ice_fields(s, args):
    x = xr.open_mfdataset(args.save_dir + s + "/data_20*_sea_ice_input_normalized.nc")
    return x

def open_atmo_fields(s, args):
    x = xr.open_mfdataset(args.save_dir + s + "/data_20*_atmo_input_normalized.nc")
    return x

def open_ocean_fields(s, args):
    x = xr.open_mfdataset(args.save_dir + s + "/data_20*_ocean_input_normalized.nc")
    return x
def open_ocean_forcings_fields(s, args):
    x = xr.open_mfdataset(args.save_dir + s + "/data_20*_ocean_forcings_input_normalized.nc")
    return x
def open_sea_ice_fields_output(s, args):
    x = xr.open_mfdataset(args.save_dir + s + "/data_20*_sea_ice_output_normalized.nc")
    return x


def open_ocean_fields_output(s, args):
    x = xr.open_mfdataset(args.save_dir + s + "/data_20*_ocean_output_normalized.nc")
    return x
def merge_fields_inputs(args):
    stage = ["train", "val", "test"]
    for s in stage: 
        print(s)
        x = open_sea_ice_fields(s, args)
        print(x)
        y = open_atmo_fields(s, args)
        print(y)
        y2 = open_winds_fields(s, args)
        print(y2)
        z = open_ocean_fields(s, args)
        #print(z)
        z2 = open_ocean_forcings_fields(s, args)
        y = y.isel(time = slice(0, -args.timestep_difference))
        y2 = y2.isel(time_counter = slice(0, -args.timestep_difference))
        
        #print(z2)
        #y = y.resample(time='1H', label='left').mean()
        y = y.assign_coords(time=y.time - pd.Timedelta('30min'))
        y2= y2.rename({'time_counter': 'time'})
        #y2 = y2.assign_coords(time=y.time - pd.Timedelta('30min'))
        #y2= y2.rename({'time_counter': 'time'})
        
        y["time"] = y2["time"]
        x_years = x.groupby('time.year')
        y_years = y.groupby('time.year')
        y2_years = y2.groupby('time.year')
        z_years = z.groupby('time.year')
        z2_years = z2.groupby('time.year')
        # Find common years
        x_year_groups = dict(list(x_years))
        y_year_groups = dict(list(y_years))
        y2_year_groups = dict(list(y2_years))
        z_year_groups = dict(list(z_years))
        z2_year_groups = dict(list(z2_years))
        common_years = set(x_year_groups.keys()).intersection(set(y_year_groups.keys()))
                
        # Process each year
        for year in common_years:
            print(year)
            # Get data for current year
            x_year_data = x_year_groups[year]
            y_year_data = y_year_groups[year]
            y_year_data['time'] = x_year_data['time']
            y2_year_data = y2_year_groups[year]
            y2_year_data['time'] = x_year_data['time']
            z_year_data = z_year_groups[year]
            z2_year_data = z2_year_groups[year]
            # Merge the data for this specific year
            merged_data = xr.merge([x_year_data, y_year_data, y2_year_data, z_year_data, z2_year_data], join = 'outer', compat='override')
            #merged_data = xr.merge([x_year_data, y_year_data, y2_year_data], join = 'outer', compat='override')

            merged_data.sel(time=x_year_data.time.values)
            # Save the merged data
            filename_output = args.save_dir + f'{s}/data_{year}_merged.nc'
            merged_data.to_netcdf(filename_output)
            merged_data.close()


def np_to_tfrecords(X, Y, file_path_prefix, verbose=True):
    """
    Converts a Numpy array (or two Numpy arrays) into a tfrecord file.
    For supervised learning, feed training inputs to X and training labels to Y.
    For unsupervised learning, only feed training inputs to X, and feed None to Y.
    The length of the first dimensions of X and Y should be the number of samples.
    
    Parameters
    ----------
    X : numpy.ndarray of rank 2
        Numpy array for training inputs. Its dtype should be float32, float64, or int64.
        If X has a higher rank, it should be rshape before fed to this function.
    Y : numpy.ndarray of rank 2 or None
        Numpy array for training labels. Its dtype should be float32, float64, or int64.
        None if there is no label array.
    file_path_prefix : str
        The path and name of the resulting tfrecord file to be generated, without '.tfrecords'
    verbose : bool
        If true, progress is reported.
    
    Raises
    ------
    ValueError
        If input type is not float (64 or 32) or int.
    
    """

    def _dtype_feature(ndarray):
        """match appropriate tf.train.Feature class with dtype of ndarray. """
        assert isinstance(ndarray, np.ndarray)
        dtype_ = ndarray.dtype
        if dtype_ == np.float64 or dtype_ == np.float32:
            return lambda array: tf.train.Feature(
                float_list=tf.train.FloatList(value=array)
            )
        elif dtype_ == np.int64:
            return lambda array: tf.train.Feature(
                int64_list=tf.train.Int64List(value=array)
            )
        else:
            raise ValueError(
                "The input should be numpy ndarray. \
                               Instaed got {}".format(
                    ndarray.dtype
                )
            )

    assert isinstance(X, np.ndarray)
    assert len(X.shape) == 2  # If X has a higher rank,
    # it should be rshape before fed to this function.
    assert isinstance(Y, np.ndarray) or Y is None

    # load appropriate tf.train.Feature class depending on dtype
    dtype_feature_x = _dtype_feature(X)

    assert X.shape[0] == Y.shape[0]
    assert len(Y.shape) == 2
    dtype_feature_y = _dtype_feature(Y)

    # Generate tfrecord writer
    result_tf_file = file_path_prefix + ".tfrecords"
    writer = tf.io.TFRecordWriter(result_tf_file)
    if verbose:
        print("Serializing {:d} examples into {}".format(X.shape[0], result_tf_file))

    # iterate over each sample,
    # and serialize it as ProtoBuf.
    for idx in trange(X.shape[0]):
        x = X[idx]

        y = Y[idx]

        d_feature = {}
        d_feature["inputs"] = dtype_feature_x(x)

        d_feature["outputs"] = dtype_feature_y(y)

        features = tf.train.Features(feature=d_feature)
        example = tf.train.Example(features=features)
        serialized = example.SerializeToString()
        writer.write(serialized)
    if verbose:
        print("Writing {} done!".format(result_tf_file))


def split_tfrecord(tfrecord_path, split_size):
    with tf.Graph().as_default(), tf.Session() as sess:
        ds = tf.data.TFRecordDataset(tfrecord_path).batch(split_size)
        batch = ds.make_one_shot_iterator().get_next()
        part_num = 0
        while True:
            try:
                records = sess.run(batch)
                part_path = tfrecord_path + ".{:03d}".format(part_num)
                with tf.python_io.TFRecordWriter(part_path) as writer:
                    for record in records:
                        writer.write(record)
                part_num += 1
            except tf.errors.OutOfRangeError:
                break

def wrap_tfrecords(args):
    years_train = range(2017, args.train_year_end + 1)
    #years_train = range(args.train_year_begin, 2016 + 1)
    years_val = range(args.val_year_begin, args.val_year_end + 1)
    #years_test = range(2020, args.test_year_end + 1)
    years_test = range(2019, args.test_year_end)

    #move_to_tfrecords(args, "train", years_train)
    #move_to_tfrecords(args, "val", years_val)
    move_to_tfrecords(args, "test", years_test)

def move_to_tfrecords(args, stage, years):
    for year in years:
        print(year)
        x = xr.open_dataset(args.save_dir + stage + f"/data_{year:d}_merged.nc")
        y = xr.open_dataset(args.save_dir + stage + f"/data_{year:d}_sea_ice_output_normalized.nc")
        #y.sel(time=x.time.values)

        x_data = x.to_array().to_numpy()#[:,:-6]
        print(np.shape(x_data))
        y_data = y.to_array().to_numpy()
        x_data = np.swapaxes(x_data,0, 1)
        print(np.shape(y_data))

        y_data = np.swapaxes(y_data,0, 1)
     
        x_data = x_data.reshape((np.shape(x_data)[0],-1))
        y_data = y_data.reshape((np.shape(y_data)[0],-1))
        
        new_path = args.save_dir + stage + f"/data_{year}"
        np_to_tfrecords(x_data, y_data, new_path, verbose=True)
        split_tfrecord(args.save_dir + stage + f"/data_{year}.tfrecords", 60)
        x.close()
        y.close()

In [7]:
merge_fields_inputs(args)


train
<xarray.Dataset>
Dimensions:    (time: 70026, prec: 1, x: 128, y: 128)
Coordinates:
  * time       (time) datetime64[ns] 2010-01-01T06:30:00 ... 2017-12-31T11:30:00
  * prec       (prec) int64 0
  * x          (x) int64 0 1 2 3 4 5 6 7 8 ... 120 121 122 123 124 125 126 127
  * y          (y) int64 0 1 2 3 4 5 6 7 8 ... 120 121 122 123 124 125 126 127
    latitude   (y, x) float64 dask.array<chunksize=(128, 128), meta=np.ndarray>
    longitude  (y, x) float64 dask.array<chunksize=(128, 128), meta=np.ndarray>
Data variables:
    sivolu     (time, prec, y, x) float32 dask.array<chunksize=(8748, 1, 128, 128), meta=np.ndarray>
    siconc     (time, prec, y, x) float32 dask.array<chunksize=(8748, 1, 128, 128), meta=np.ndarray>
    u_ice      (time, prec, y, x) float32 dask.array<chunksize=(8748, 1, 128, 128), meta=np.ndarray>
    v_ice      (time, prec, y, x) float32 dask.array<chunksize=(8748, 1, 128, 128), meta=np.ndarray>
    snvolu     (time, prec, y, x) float32 dask.array<chunksiz

In [15]:
wrap_tfrecords(args)

2019
(17, 8748, 1, 128, 128)
(5, 8748, 1, 128, 128)
Serializing 8748 examples into /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/test/data_2019.tfrecords


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8748/8748 [10:03<00:00, 14.49it/s]


Writing /summer/meom/workdir/ducharlo/dataset/nanuk1_6h_with_ocean_under/test/data_2019.tfrecords done!


In [16]:
args_sic= ['sit', 'sic', 'u_ice', 'v_ice', 'snvolu']